# RetinaScreen AI: Hybrid Ensemble (EfficientNet + ViT)

This notebook trains an advanced ensemble model using separate CNN (EfficientNet-B4) and Vision Transformer (ViT-B16) models, and then finds the optimal weighting for the final predictions.

## 1. Setup Environment

In [ ]:
#SETUP + IMPORTS
!pip install -q keras-cv

import os, json, glob
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt
import keras_cv

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

search_paths = glob.glob('/kaggle/input/**/train', recursive=True)

if search_paths:
    BASE_DIR = os.path.dirname(search_paths[0])
    print(f"Found dataset at: {BASE_DIR}")
else:
    BASE_DIR = '/kaggle/input/diabetic-retinopathy-dataset/Diabetic_Retinopathy_dataset'
    print(f"Using fallback path: {BASE_DIR}")

TRAIN_ORIGINAL = os.path.join(BASE_DIR, 'train/original')
TEST_ORIGINAL = os.path.join(BASE_DIR, 'test/original')

TRAIN_CLAHE = os.path.join(BASE_DIR, 'train/clahe')
TEST_CLAHE = os.path.join(BASE_DIR, 'test/clahe')

MODELS_OUTPUT_DIR = '/kaggle/working/models_output'
os.makedirs(MODELS_OUTPUT_DIR, exist_ok=True)

print("Setup complete.")

## 2. Dataset Processing

In [ ]:
# CLEAN + LOAD DATASET

import shutil

CLEAN_BASE = "/kaggle/working/clean_dataset"

if os.path.exists(CLEAN_BASE):
    shutil.rmtree(CLEAN_BASE)


def find_corrupted_images(base_path):

    corrupted = []

    if not os.path.exists(base_path):
        return corrupted

    print(f"Scanning: {base_path}")

    for root, dirs, files in os.walk(base_path):

        for file in files:

            file_path = os.path.join(root, file)

            try:
                image_content = tf.io.read_file(file_path)

                tf.image.decode_image(
                    image_content,
                    channels=3,
                    expand_animations=False
                )

            except Exception:

                corrupted.append(file_path)
                print(f"Corrupted: {file_path}")

    print(f"Total corrupted images: {len(corrupted)}\n")

    return corrupted


def copy_valid_dataset(source_dir, destination_dir, corrupted_files):

    corrupted_set = set(corrupted_files)

    for class_name in os.listdir(source_dir):

        source_class = os.path.join(
            source_dir,
            class_name
        )

        destination_class = os.path.join(
            destination_dir,
            class_name
        )

        if not os.path.isdir(source_class):
            continue

        os.makedirs(
            destination_class,
            exist_ok=True
        )

        for file in os.listdir(source_class):

            source_file = os.path.join(
                source_class,
                file
            )

            if source_file in corrupted_set:
                continue

            destination_file = os.path.join(
                destination_class,
                file
            )

            shutil.copy2(
                source_file,
                destination_file
            )


def load_datasets(train_dir, test_dir):

    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        subset="training",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir,
        seed=SEED,
        shuffle=False,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    return train_ds, val_ds, test_ds


CORRUPTED_TRAIN = find_corrupted_images(TRAIN_ORIGINAL)
CORRUPTED_TEST = find_corrupted_images(TEST_ORIGINAL)

CLEAN_TRAIN = os.path.join(
    CLEAN_BASE,
    "train"
)

CLEAN_TEST = os.path.join(
    CLEAN_BASE,
    "test"
)

os.makedirs(CLEAN_TRAIN, exist_ok=True)
os.makedirs(CLEAN_TEST, exist_ok=True)


copy_valid_dataset(
    TRAIN_ORIGINAL,
    CLEAN_TRAIN,
    CORRUPTED_TRAIN
)

copy_valid_dataset(
    TEST_ORIGINAL,
    CLEAN_TEST,
    CORRUPTED_TEST
)


train_ds, val_ds, test_ds = load_datasets(
    CLEAN_TRAIN,
    CLEAN_TEST
)


CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)


AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


print("\nClasses:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)
print("Corrupted train images:", len(CORRUPTED_TRAIN))
print("Corrupted test images:", len(CORRUPTED_TEST))
print("Clean dataset:", CLEAN_BASE)

## 3. Train EfficientNet-B4 (CNN Branch)

In [ ]:
#EFFICIENTNET_B4: Build + train + fine_tune
def build_efficientnet_b4():

    base_model = EfficientNetB4(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )

    base_model.trainable = False

    inputs = tf.keras.Input(
        shape=(*IMG_SIZE, 3)
    )

    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model, base_model


efficientnet_model, efficientnet_base = build_efficientnet_b4()

EFFICIENTNET_PATH = os.path.join(
    MODELS_OUTPUT_DIR,
    "efficientnet_b4_best.keras"
)

eff_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    callbacks.ModelCheckpoint(
        EFFICIENTNET_PATH,
        monitor="val_loss",
        save_best_only=True
    )
]

print("--- EfficientNet-B4 Phase 1 ---")

history_eff_head = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=eff_callbacks
)

print("\n--- EfficientNet-B4 Phase 2 ---")

efficientnet_base.trainable = True

for layer in efficientnet_base.layers[:-40]:
    layer.trainable = False

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_eff_finetune = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=eff_callbacks
)

## 4. Train ViT-B/16 (Transformer Branch)

In [ ]:
# VIT-B/16: Build + train + fine-tune
!pip install -q -U keras-hub
import keras_hub 
VIT_PRESET = "vit_base_patch16_224_imagenet"


def build_vit_b16():

    vit_base = keras_hub.models.ViTBackbone.from_preset(
        VIT_PRESET
    )

    vit_preprocessor = (
        keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
            VIT_PRESET
        )
    )

    vit_base.trainable = False

    inputs = tf.keras.Input(
        shape=(*IMG_SIZE, 3),
        name="image"
    )

    x = vit_preprocessor(inputs)

    x = vit_base(x)

    x = layers.Cropping1D(
        cropping=(0, 196),
        name="extract_cls_token"
    )(x)

    x = layers.Flatten(
        name="cls_embedding"
    )(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name="ViT_B16_DR"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model, vit_base


vit_model, vit_base = build_vit_b16()


VIT_PATH = os.path.join(
    MODELS_OUTPUT_DIR,
    "vit_b16_best.keras"
)


vit_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    callbacks.ModelCheckpoint(
        VIT_PATH,
        monitor="val_loss",
        save_best_only=True
    ),

    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7
    )
]


print("--- ViT-B/16 Phase 1 ---")

history_vit_head = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=vit_callbacks
)


print("\n--- ViT-B/16 Phase 2 ---")

vit_base.trainable = True

for layer in vit_base.layers[:-20]:

    layer.trainable = False


vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


history_vit_finetune = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=vit_callbacks
)


print("\nViT-B/16 training completed.")
print("Best model saved to:", VIT_PATH)

## 5. Evaluate Ensemble & Optimize Weights

In [ ]:
# EVALUATION + ENSEMBLE

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

efficientnet_model = tf.keras.models.load_model(
    EFFICIENTNET_PATH
)

vit_model = tf.keras.models.load_model(
    VIT_PATH
)

print("\n--- Validation Predictions ---")

eff_val_pred = efficientnet_model.predict(
    val_ds,
    verbose=1
)

vit_val_pred = vit_model.predict(
    val_ds,
    verbose=1
)

y_val_true = np.concatenate([
    np.argmax(y.numpy(), axis=1)
    for _, y in val_ds
])

best_eff_weight = 0.5
best_vit_weight = 0.5
best_f1 = -1.0

for eff_weight in np.arange(0.0, 1.01, 0.05):

    vit_weight = 1.0 - eff_weight

    ensemble_val_pred = (
        eff_weight * eff_val_pred +
        vit_weight * vit_val_pred
    )

    ensemble_val_classes = np.argmax(
        ensemble_val_pred,
        axis=1
    )

    score = f1_score(
        y_val_true,
        ensemble_val_classes,
        average="macro"
    )

    if score > best_f1:

        best_f1 = score
        best_eff_weight = eff_weight
        best_vit_weight = vit_weight


print("\nBest Ensemble Weights:")
print("EfficientNet-B4:", best_eff_weight)
print("ViT-B/16:", best_vit_weight)
print("Validation Macro F1:", f"{best_f1:.4f}")


# --------------------------------------------------
# TEST PREDICTIONS
# --------------------------------------------------

print("\n--- Test Predictions ---")

eff_test_pred = efficientnet_model.predict(
    test_ds,
    verbose=1
)

vit_test_pred = vit_model.predict(
    test_ds,
    verbose=1
)

y_test_true = np.concatenate([
    np.argmax(y.numpy(), axis=1)
    for _, y in test_ds
])


# --------------------------------------------------
# ENSEMBLE PREDICTIONS
# --------------------------------------------------

ensemble_test_pred = (
    best_eff_weight * eff_test_pred +
    best_vit_weight * vit_test_pred
)


y_eff_pred = np.argmax(
    eff_test_pred,
    axis=1
)

y_vit_pred = np.argmax(
    vit_test_pred,
    axis=1
)

y_ensemble_pred = np.argmax(
    ensemble_test_pred,
    axis=1
)


# --------------------------------------------------
# SAVE ALL PREDICTIONS
# --------------------------------------------------

EFF_PRED_PATH = os.path.join(
    MODELS_OUTPUT_DIR,
    "efficientnet_b4_test_predictions.npy"
)

VIT_PRED_PATH = os.path.join(
    MODELS_OUTPUT_DIR,
    "vit_b16_test_predictions.npy"
)

ENSEMBLE_PRED_PATH = os.path.join(
    MODELS_OUTPUT_DIR,
    "ensemble_test_predictions.npy"
)

TRUE_LABELS_PATH = os.path.join(
    MODELS_OUTPUT_DIR,
    "test_true_labels.npy"
)

np.save(
    EFF_PRED_PATH,
    eff_test_pred
)

np.save(
    VIT_PRED_PATH,
    vit_test_pred
)

np.save(
    ENSEMBLE_PRED_PATH,
    ensemble_test_pred
)

np.save(
    TRUE_LABELS_PATH,
    y_test_true
)


# --------------------------------------------------
# SAVE ENSEMBLE PREDICTION CSV
# --------------------------------------------------

ensemble_prediction_data = {
    "true_class": [
        CLASS_NAMES[i]
        for i in y_test_true
    ],

    "predicted_class": [
        CLASS_NAMES[i]
        for i in y_ensemble_pred
    ]
}

for i, class_name in enumerate(CLASS_NAMES):

    ensemble_prediction_data[
        f"prob_{class_name}"
    ] = ensemble_test_pred[:, i]


ensemble_predictions_df = pd.DataFrame(
    ensemble_prediction_data
)

ENSEMBLE_CSV_PATH = os.path.join(
    MODELS_OUTPUT_DIR,
    "ensemble_test_predictions.csv"
)

ensemble_predictions_df.to_csv(
    ENSEMBLE_CSV_PATH,
    index=False
)


print("\nPredictions saved:")
print("EfficientNet predictions:", EFF_PRED_PATH)
print("ViT predictions:", VIT_PRED_PATH)
print("Ensemble predictions:", ENSEMBLE_PRED_PATH)
print("True labels:", TRUE_LABELS_PATH)
print("Ensemble CSV:", ENSEMBLE_CSV_PATH)


# --------------------------------------------------
# EFFICIENTNET RESULTS
# --------------------------------------------------

print("\n==============================")
print("EFFICIENTNET-B4")
print("==============================")

eff_accuracy = accuracy_score(
    y_test_true,
    y_eff_pred
)

eff_macro_f1 = f1_score(
    y_test_true,
    y_eff_pred,
    average="macro"
)

print("Accuracy:", f"{eff_accuracy:.4f}")
print("Macro F1:", f"{eff_macro_f1:.4f}")

print(
    classification_report(
        y_test_true,
        y_eff_pred,
        target_names=CLASS_NAMES,
        digits=4
    )
)


# --------------------------------------------------
# VIT RESULTS
# --------------------------------------------------

print("\n==============================")
print("VIT-B/16")
print("==============================")

vit_accuracy = accuracy_score(
    y_test_true,
    y_vit_pred
)

vit_macro_f1 = f1_score(
    y_test_true,
    y_vit_pred,
    average="macro"
)

print("Accuracy:", f"{vit_accuracy:.4f}")
print("Macro F1:", f"{vit_macro_f1:.4f}")

print(
    classification_report(
        y_test_true,
        y_vit_pred,
        target_names=CLASS_NAMES,
        digits=4
    )
)


# --------------------------------------------------
# ENSEMBLE RESULTS
# --------------------------------------------------

print("\n==============================")
print("EFFICIENTNET-B4 + VIT-B/16")
print("ENSEMBLE")
print("==============================")

ensemble_accuracy = accuracy_score(
    y_test_true,
    y_ensemble_pred
)

ensemble_macro_f1 = f1_score(
    y_test_true,
    y_ensemble_pred,
    average="macro"
)

print("Accuracy:", f"{ensemble_accuracy:.4f}")
print("Macro F1:", f"{ensemble_macro_f1:.4f}")

print(
    classification_report(
        y_test_true,
        y_ensemble_pred,
        target_names=CLASS_NAMES,
        digits=4
    )
)


# --------------------------------------------------
# FINAL COMPARISON
# --------------------------------------------------

print("\n==============================")
print("FINAL COMPARISON")
print("==============================")

print(
    f"EfficientNet-B4 : "
    f"Accuracy={eff_accuracy:.4f} | "
    f"F1={eff_macro_f1:.4f}"
)

print(
    f"ViT-B/16        : "
    f"Accuracy={vit_accuracy:.4f} | "
    f"F1={vit_macro_f1:.4f}"
)

print(
    f"Ensemble        : "
    f"Accuracy={ensemble_accuracy:.4f} | "
    f"F1={ensemble_macro_f1:.4f}"
)


# --------------------------------------------------
# CREATE STANDALONE ENSEMBLE MODEL
# --------------------------------------------------

ensemble_input = tf.keras.Input(
    shape=(*IMG_SIZE, 3),
    name="image"
)

eff_output = efficientnet_model(
    ensemble_input,
    training=False
)

vit_output = vit_model(
    ensemble_input,
    training=False
)

eff_weighted = layers.Multiply(
    name="efficientnet_weight"
)([
    eff_output,
    tf.constant(
        float(best_eff_weight),
        dtype=tf.float32
    )
])

vit_weighted = layers.Multiply(
    name="vit_weight"
)([
    vit_output,
    tf.constant(
        float(best_vit_weight),
        dtype=tf.float32
    )
])

ensemble_output = layers.Add(
    name="weighted_probability_average"
)([
    eff_weighted,
    vit_weighted
])

ensemble_model = models.Model(
    ensemble_input,
    ensemble_output,
    name="EfficientNetB4_ViTB16_Ensemble"
)

ENSEMBLE_PATH = os.path.join(
    MODELS_OUTPUT_DIR,
    "ensemble_effnet_vit_best.keras"
)

ensemble_model.save(
    ENSEMBLE_PATH
)

print("\nEnsemble model saved:")
print(ENSEMBLE_PATH)


# --------------------------------------------------
# CONFUSION MATRIX
# --------------------------------------------------

cm = confusion_matrix(
    y_test_true,
    y_ensemble_pred
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

disp.plot(
    cmap="Blues",
    ax=ax,
    values_format="d",
    xticks_rotation="vertical"
)

plt.title(
    "EfficientNet-B4 + ViT-B/16 Ensemble"
)

plt.tight_layout()
plt.show()

## 6. Save Config & Models

In [ ]:
# SAVE CONFIGURATION

config = {
    "model_name": "ensemble_efficientnet_b4_vit_b16",

    "image_size": list(IMG_SIZE),

    "batch_size": BATCH_SIZE,

    "class_names": CLASS_NAMES,

    "class_mapping": {
        c: i for i, c in enumerate(CLASS_NAMES)
    },

    "num_classes": NUM_CLASSES,

    "dataset": "original",

    "models": [
        "EfficientNet-B4",
        "ViT-B/16"
    ],

    "vit_preset": "vit_base_patch16_224_imagenet",

    "ensemble_method": "weighted_probability_average",

    "efficientnet_weight": float(best_eff_weight),

    "vit_weight": float(best_vit_weight),

    "output_activation": "softmax",

    "efficientnet_model": "efficientnet_b4_best.keras",

    "vit_model": "vit_b16_best.keras",

    "ensemble_model": "ensemble_effnet_vit_best.keras",

    "test_results": {
        "efficientnet_b4": {
            "accuracy": float(eff_accuracy),
            "macro_f1": float(eff_macro_f1)
        },

        "vit_b16": {
            "accuracy": float(vit_accuracy),
            "macro_f1": float(vit_macro_f1)
        },

        "ensemble": {
            "accuracy": float(ensemble_accuracy),
            "macro_f1": float(ensemble_macro_f1)
        }
    }
}

config_path = os.path.join(
    MODELS_OUTPUT_DIR,
    "ensemble_efficientnet_b4_vit_b16_config.json"
)

with open(config_path, "w") as f:

    json.dump(
        config,
        f,
        indent=4
    )

print("Config saved:", config_path)

print("\nFINAL FILES:")

print("EfficientNet:", EFFICIENTNET_PATH)
print("ViT:", VIT_PATH)
print("Config:", config_path)
print("Ensemble:", ENSEMBLE_PATH)
